# Part 6: Ensemble Models & Full Comparison
## GNN + ML Ensemble and Complete Model Ranking

This notebook:
1. Trains **GNNEnsembleClassifier** (GNN embeddings + fingerprints → XGBoost/RF)
2. Compares ALL models: ML baselines, GNNs, advanced GNNs, ensembles
3. Produces the final ranking table

In [ ]:
# @title 1. Setup
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("No GPU")

train_df = pd.read_csv('data/train.csv')
val_df = pd.read_csv('data/val.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# @title 2. Train GNN + ML Ensembles
from vegfr2.models.ensemble import GNNEnsembleClassifier
from vegfr2.metrics import classification_metrics

ensemble_configs = [
    ('gin', 'xgb'),
    ('pna', 'xgb'),
    ('gin', 'rf'),
    ('pna', 'rf'),
]

results = {}

for gnn_name, ml_name in ensemble_configs:
    name = f'ensemble_{gnn_name}_{ml_name}'
    print(f"\nTraining {name}...")
    
    try:
        ensemble = GNNEnsembleClassifier(
            gnn_name=gnn_name,
            ml_name=ml_name,
            hidden=128,
            layers=3,
            heads=8,
            dropout=0.3,
            seed=42,
        )
        
        ensemble.fit(
            train_smiles=train_df['smiles'].tolist(),
            train_labels=train_df['active'].astype(int).tolist(),
            val_smiles=val_df['smiles'].tolist(),
            val_labels=val_df['active'].astype(int).tolist(),
            device=DEVICE,
            gnn_epochs=50,
        )
        
        probs = ensemble.predict_proba(test_df['smiles'].tolist(), device=DEVICE)
        y_test = test_df['active'].values.tolist()
        metrics = classification_metrics(y_test, probs.tolist())
        results[name] = metrics
        
        print(f"  AUC={metrics.get('auc', 0):.4f} ACC={metrics['acc']:.4f} MCC={metrics['mcc']:.4f}")
        
        # Save
        save_dir = Path(f'models/{name}')
        save_dir.mkdir(parents=True, exist_ok=True)
        ensemble.save(str(save_dir / 'ensemble.pkl'))
        
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
# @title 3. Load Previous Results
# Load ML results
try:
    ml_results = pd.read_csv('../Part_3/data/ml_results.csv')
    for _, row in ml_results.iterrows():
        results[row['model']] = {
            'acc': row['acc'], 'sen': row['sen'], 'spe': row['spe'],
            'mcc': row['mcc'], 'auc': row['auc']
        }
    print(f"Loaded {len(ml_results)} ML results")
except Exception as e:
    print(f"ML results not found: {e}")

# Load GNN results (from Part 4)
# These would be loaded from saved checkpoints
try:
    gnn_results = pd.read_csv('../Part_4/data/gnn_results.csv')
    for _, row in gnn_results.iterrows():
        results[row['model']] = {
            'acc': row['acc'], 'sen': row['sen'], 'spe': row['spe'],
            'mcc': row['mcc'], 'auc': row['auc']
        }
    print(f"Loaded {len(gnn_results)} GNN results")
except Exception as e:
    print(f"GNN results not found: {e}")

In [ ]:
# @title 4. Complete Comparison Table
print("\n" + "=" * 80)
print("COMPLETE VEGFR2 MODEL COMPARISON")
print("=" * 80)

# Group by type
ml_results_dict = {k: v for k, v in results.items() if k.startswith(('rf_', 'svm_', 'xgb_'))}
gnn_results_dict = {k: v for k, v in results.items() if k.startswith(('gcn', 'gat', 'gin', 'pna', 'mpnn', 'graph_transformer', 'attentive_fp'))}
ensemble_results_dict = {k: v for k, v in results.items() if k.startswith('ensemble_')}

print(f"\n{'CLASSICAL ML':^80}")
print("-" * 80)
print(f"{'Model':<30} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}")
for name, m in sorted(ml_results_dict.items(), key=lambda x: x[1].get('auc') or 0, reverse=True):
    auc_str = f"{m['auc']:.4f}" if m.get('auc') is not None else "N/A"
    print(f"{name:<30} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

print(f"\n{'GNN MODELS':^80}")
print("-" * 80)
print(f"{'Model':<30} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}")
for name, m in sorted(gnn_results_dict.items(), key=lambda x: x[1].get('auc') or 0, reverse=True):
    auc_str = f"{m['auc']:.4f}" if m.get('auc') is not None else "N/A"
    print(f"{name:<30} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

print(f"\n{'ENSEMBLE MODELS':^80}")
print("-" * 80)
print(f"{'Model':<30} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}")
for name, m in sorted(ensemble_results_dict.items(), key=lambda x: x[1].get('auc') or 0, reverse=True):
    auc_str = f"{m['auc']:.4f}" if m.get('auc') is not None else "N/A"
    print(f"{name:<30} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

# Overall ranking
print(f"\n{'OVERALL TOP 10':^80}")
print("-" * 80)
all_sorted = sorted(results.items(), key=lambda x: x[1].get('auc') or 0, reverse=True)
for i, (name, m) in enumerate(all_sorted[:10], 1):
    auc_str = f"{m['auc']:.4f}" if m.get('auc') is not None else "N/A"
    print(f"  {i}. {name:<35} AUC={auc_str}")

In [ ]:
# @title 5. Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 10 models
top10 = all_sorted[:10]
names = [n for n, _ in top10]
aucs = [m.get('auc', 0) for _, m in top10]
colors = ['#2196F3' if 'ensemble' in n else '#4CAF50' if 'gnn' in n or 'gcn' in n or 'gat' in n or 'gin' in n or 'pna' in n else '#FF9800' for n in names]

axes[0].barh(range(len(names)), aucs, color=colors)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names)
axes[0].set_xlabel('AUC')
axes[0].set_title('Top 10 Models by AUC')
axes[0].set_xlim(0.5, 1.0)

# Category averages
categories = {
    'ML (Morgan)': [v.get('auc', 0) for k, v in results.items() if 'morgan' in k and 'gnn' not in k],
    'ML (MACCS)': [v.get('auc', 0) for k, v in results.items() if 'maccs' in k and 'gnn' not in k],
    'GNN (Enriched)': [v.get('auc', 0) for k, v in results.items() if 'enriched' in k],
    'Ensemble': [v.get('auc', 0) for k, v in results.items() if 'ensemble' in k],
}

cat_names = list(categories.keys())
cat_aucs = [np.mean(v) if v else 0 for v in categories.values()]
axes[1].bar(cat_names, cat_aucs, color=['#FF9800', '#9C27B0', '#4CAF50', '#2196F3'])
axes[1].set_ylabel('Average AUC')
axes[1].set_title('Average AUC by Category')
axes[1].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig('images/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# @title 6. Save Final Results
import json

output = {
    'results': {k: {kk: float(vv) if isinstance(vv, (np.floating, float)) else vv 
                     for kk, vv in v.items() if kk != 'confusion_matrix'}
                for k, v in results.items()}
}

with open('data/final_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(results)} model results to data/final_results.json")